# GridNet-HD — Leaderboard: ทำนาย test 9 zones แล้วสร้างไฟล์ส่ง (LiDAR-only + fused model)

**ต้องมีก่อน**
1. LiDAR cache ของ test 9 zones บน Drive (`GridNet/cache/<zone>/points.npz`) — สร้างด้วย `colab_build_cache` บน **CPU High-RAM** (ไม่ใช้ GPU; ต้องใช้โค้ดของ update35 ขึ้นไป)
2. checkpoint ทั้งสาม: LiDAR-only (`ckpt/litept_s_lidar_v2/best.pth`), UPerNet (`ckpt/upernet_convnext_t_v2/best.pth`), head (`fusion/late_a/best.pth`)
3. `GridNet/code/gridnet_code.zip` ของ update35 ขึ้นไป

**ทำอะไร** (ทีละโซน, resume ได้): โหลดภาพ + pose ของโซนจาก Hugging Face → export 2D cache ลง local disk → association บน GPU →
ทำนายด้วย frozen LitePT-S + head (forward ชุดเดียวกันให้ทั้ง **fused** และ **LiDAR-only**) → ส่ง class ของ voxel กลับไปที่จุดดิบทุกจุดตามลำดับใน LAS →
`GridNet/leaderboard/<TAG>/fused/<zone>.npz` และ `.../lidar/<zone>.npz` (array `data`, uint8, class 0–10) → ลบไฟล์ local ของโซนนั้น
2D cache ของ test zones ไม่ถูกเก็บลง Drive (ใช้ครั้งเดียว; ราว 24 MB ต่อภาพ)

**ตรวจก่อนเริ่ม** (cell 6, `CHECK_ZONE`): val zone หนึ่งโซนถูกรันผ่านเส้นทางเดียวกันทั้งหมด แล้วเทียบกับผล validation ที่มีอยู่
(ต้องได้ mIoU เท่าเดิมภายใน `CHECK_TOLERANCE`) + export 2D ใหม่ 6 ภาพเทียบกับ cache เดิม + วัด mIoU บน **จุดดิบ** จาก LAS
(leaderboard วัดบนจุดดิบ ส่วนตัวเลข validation ของเราวัดบน voxel 5 cm) — ถ้าไม่ผ่าน notebook หยุดก่อนถึง test zones

**Runtime**: A100 = ค่าที่ใช้กับตัวเลข validation ทุกตัว (window 900,000 จุด) · L4 + High-RAM ใช้ได้ด้วย window 500,000 จุด แต่ **ยังไม่เคยวัด**: ดูผลของ cell 6 ก่อน
เวลาที่วัดได้บน A100-40GB (14 พ.ย. 2026; test = 767 ล้านจุดดิบ = 524 ล้าน voxel, 2,447 ภาพ): ตรวจ 4 นาที, test 9 zones 107 นาที = 6–22 นาทีต่อโซน
(export 2D 1.4 s/ภาพ จึงขึ้นกับจำนวนภาพ: สองโซนของพื้นที่ t6 มีโซนละกว่า 600 ภาพ ใช้โซนละ 21 นาที; ทำนาย 2.5–6.7 นาทีต่อโซน) → ทั้ง notebook ราว 1.9 ชม.

**ส่งจริง**: ตรวจผลของ notebook ที่รันแล้วก่อน แล้วอัปโหลดไฟล์ `.npz` ทั้ง 9 ไฟล์ของโฟลเดอร์ `lidar/` (sanity check) และ `fused/` ที่
https://huggingface.co/spaces/heig-vd-geo/GridNet-HD-Leaderboard → Sign in → Submit Eval · **ไม่ติ๊ก** "Remap classes" (ไฟล์เป็น class 0–10 ที่ remap แล้ว)

In [ ]:
#@title 1. GPU (A100 = the setting of every validation number; an L4 works with smaller windows - see CHECK_ZONE)
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8, "need an A100 or an L4 (flash-attn)"
VRAM_MB = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split()[0])
print("windows of 900,000 points (the validation setting)" if VRAM_MB >= 30000 else
      "smaller GPU: windows of 500,000 points - NOT the validation setting; cell 6 shows what that changes on a validation zone")


In [ ]:
#@title 2. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
TAG = 'v1'                                       #@param {type:"string"}
HEAD = 'late_a'                                  #@param {type:"string"}
RUN2D = 'upernet_convnext_t_v2'                  #@param {type:"string"}
BACKBONE_CKPT = f'{DRIVE}/ckpt/litept_s_lidar_v2/best.pth'   #@param {type:"string"}
ZONES = 't1z4,t1z5a,t1z7,t3z1,t3z2,t3z5,t5a2,t6z1,t6z5'      #@param {type:"string"}
# a validation zone that goes through the same path first ('' = skip); t1z8 is the smallest and has the largest fusion gain
CHECK_ZONE = 't1z8'                              #@param {type:"string"}
CHECK_TOLERANCE = 0.5                            #@param {type:"number"}
EVAL_MAX_POINTS = 0                              #@param {type:"integer"}
EXPORT_BATCH = 0                                 #@param {type:"integer"}
SHUTDOWN_WHEN_DONE = False                       #@param {type:"boolean"}
CACHE_DRIVE = f'{DRIVE}/cache'; CACHE2D_DRIVE = f'{DRIVE}/cache2d/{RUN2D}'; CKPT2D_DRIVE = f'{DRIVE}/ckpt/{RUN2D}'
HEAD_CKPT = f'{DRIVE}/fusion/{HEAD}/best.pth'; HEAD_EVAL = f'{DRIVE}/fusion/{HEAD}/eval'
OUT = f'{DRIVE}/leaderboard/{TAG}'
zones = [z.strip() for z in ZONES.split(',') if z.strip()]
if not EVAL_MAX_POINTS: EVAL_MAX_POINTS = 900000 if VRAM_MB >= 30000 else 500000     # 0 = by the GPU
if not EXPORT_BATCH: EXPORT_BATCH = 4 if VRAM_MB >= 30000 else 2
for p in (BACKBONE_CKPT, HEAD_CKPT, f'{CKPT2D_DRIVE}/best.pth'):
    assert os.path.exists(p), f"missing {p}"
missing = [z for z in zones + ([CHECK_ZONE] if CHECK_ZONE else []) if not os.path.exists(f'{CACHE_DRIVE}/{z}/points.npz')]
assert not missing, f"no LiDAR cache on Drive for {missing}: run colab_build_cache (CPU High-RAM runtime) first"
os.makedirs(OUT, exist_ok=True)
done = [z for z in zones if all(os.path.exists(f'{OUT}/{m}/{z}.npz') for m in ('fused', 'lidar'))]
print(f"head {HEAD}, 2D model {RUN2D}, windows of {EVAL_MAX_POINTS:,} points, 2D export batch {EXPORT_BATCH}")
print(f"output -> {OUT}/fused and {OUT}/lidar")
print(f"{len(done)}/{len(zones)} test zones have their files on Drive:", done)
print("to do:", [z for z in zones if z not in done])


In [ ]:
#@title 3. Environment (restore tarball from Drive; + timm, laspy)
import subprocess, os, time
ENV = '/content/env'; TAR = f'{DRIVE}/env/env_py310_torch241_cu124.tar.gz'
t0 = time.time()
if not os.path.exists(f'{ENV}/bin/python'):
    assert os.path.exists(TAR), "no env tarball on Drive: run colab_train_litept cell 3 once"
    subprocess.run(['tar', '-C', '/content', '-xzf', TAR], check=True); print("env restored from Drive")
PY = f'{ENV}/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', 'timm', 'huggingface_hub', 'pillow', 'laspy[lazrs]'], check=True)
print(f"env ready in {(time.time() - t0) / 60:.1f} min")


In [ ]:
#@title 4. Code + LitePT
import subprocess, zipfile, os
os.makedirs('/content/code', exist_ok=True); zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
assert os.path.exists('/content/code/scripts/predict_zones.py'), "gridnet_code.zip on Drive is older than update35: pack and upload it again"
if not os.path.exists('/content/LitePT'):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/prs-eth/LitePT.git", "/content/LitePT"], check=True)
r = subprocess.run([PY, "-c", "import sys; sys.path.insert(0,'/content/LitePT'); sys.path.insert(0,'/content/code'); "
                    "from litept.model import LitePT; import gridnet.train.predict, gridnet.train.engine2d, gridnet.submission, timm, laspy; "
                    "print('imports ok')"], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])
assert 'imports ok' in r.stdout


In [ ]:
#@title 5. Checkpoints -> local disk; helpers
import os, sys, shutil, subprocess, time, json
sys.path.insert(0, '/content/code')
os.makedirs('/content/ckpt2d', exist_ok=True)
for src, dst in ((BACKBONE_CKPT, '/content/backbone_best.pth'), (f'{CKPT2D_DRIVE}/best.pth', '/content/ckpt2d/best.pth')):
    if not os.path.exists(dst): shutil.copy(src, dst)
ENVV = dict(os.environ, PYTHONPATH='/content/code', PYTHONWARNINGS='ignore', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True',
            HF_HUB_ENABLE_HF_TRANSFER='0')
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')                  # optional (public dataset)
    if tok: ENVV['HF_TOKEN'] = tok
except Exception: pass

def run(cmd, show=lambda l: True, tag=''):
    # run a script, print the lines `show` accepts; on failure print its last lines. Returns the exit code.
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENVV)
    tail = []
    try:
        for line in p.stdout:
            tail.append(line); del tail[:-25]
            if 'Warning' in line or 'def backward' in line or '_TORCH_CUSTOM_FWD' in line: continue
            if show(line): print(tag + line, end='', flush=True)
    except KeyboardInterrupt:
        p.terminate(); p.wait(); print("stopped by user; finished zones are on Drive -> run the cell again to continue"); raise
    p.wait()
    if p.returncode != 0: print(f"{tag}exit {p.returncode}; last lines:"); print(''.join(tail[-15:]))
    return p.returncode

def fetch(zone, what, dst='/content/data'):
    # <zone>/<what>/* from the Hugging Face dataset (files already there are skipped)
    code = ("import sys; sys.path.insert(0,'/content/code'); from gridnet.train.data2d import fetch_zones; "
            f"fetch_zones(['{zone}'], '{dst}', what={tuple(what)!r})")
    return run([PY, '-c', code], show=lambda l: False)

def export_line(l):
    # of the 2D export: the first line of a zone, every 60th image and the last one
    if not l.startswith('[export]'): return 'Error' in l or 'Traceback' in l
    p = l.split()
    if len(p) > 2 and '/' in p[2]:
        k, n = p[2].split('/'); return k == n or int(k) % 60 == 0
    return True

def export2d(zone, data='/content/data', out='/content/cache2d'):
    return run([PY, '/content/code/scripts/train_upernet.py', '--data', data, '--ckpt-dir', '/content/ckpt2d', '--batch', str(EXPORT_BATCH),
                '--export-cache', out, '--export-zones', zone], show=export_line, tag='   ')

def predict(zone, out):
    return run([PY, '/content/code/scripts/predict_zones.py', '--cache', '/content/cache', '--cache2d', '/content/cache2d', '--data', '/content/data',
                '--assoc-dir', '/content/assoc', '--litept', '/content/LitePT', '--backbone-ckpt', '/content/backbone_best.pth', '--head', HEAD_CKPT,
                '--out', out, '--zones', zone, '--eval-max-points', str(EVAL_MAX_POINTS)],
               show=lambda l: l.startswith(('[predict]', '[eval]', '[fusion]')) or 'Error' in l or 'Traceback' in l, tag='   ')

def export(zone, pred_root, name, out, las_root=None):
    cmd = [PY, '/content/code/scripts/export_submission.py', '--cache', '/content/cache', '--pred', pred_root + '/{zone}/' + name + '_pred.npy',
           '--out', out, '--zones', zone, '--name', f'{name}: ' + (f'{HEAD} on {RUN2D}' if name == 'fused' else 'LiDAR-only LitePT-S') + f', windows {EVAL_MAX_POINTS}']
    if las_root: cmd += ['--las-root', las_root]
    return run(cmd, show=lambda l: l.startswith('[export]') or l.startswith('   ') or 'Error' in l or 'Traceback' in l, tag='   ')

def local_cache(zone):
    os.makedirs(f'/content/cache/{zone}', exist_ok=True)
    if not os.path.exists(f'/content/cache/{zone}/points.npz'): shutil.copy(f'{CACHE_DRIVE}/{zone}/points.npz', f'/content/cache/{zone}/points.npz')

def clean(zone):
    for d in (f'/content/data/{zone}', f'/content/cache2d/{zone}', f'/content/cache/{zone}', f'/content/assoc/{zone}'):
        shutil.rmtree(d, ignore_errors=True)
print("ready;", subprocess.run(['df', '-h', '--output=avail', '/content'], capture_output=True, text=True).stdout.split()[-1], "free on the local disk")


In [ ]:
#@title 6. Check on a validation zone (CHECK_ZONE): the same path end to end, compared with the validation run of the head (~10 min; stops the notebook when it does not reproduce)
import numpy as np
if CHECK_ZONE:
    from gridnet.train.maps2d import cached_image_ids, extract_zone_tar, zone_tar_manifest
    z, t0, check = CHECK_ZONE, time.time(), {'zone': CHECK_ZONE, 'eval_max_points': EVAL_MAX_POINTS, 'vram_mb': VRAM_MB}
    shutil.rmtree('/content/check', ignore_errors=True)
    local_cache(z)
    assert zone_tar_manifest(CACHE2D_DRIVE, z, wait=30) is not None, f"no 2D cache tar for {z} in {CACHE2D_DRIVE}"
    n = extract_zone_tar(CACHE2D_DRIVE, z, '/content/cache2d')
    assert fetch(z, ('pose', 'lidar')) == 0, "download from Hugging Face failed"
    print(f"[check {z}] LiDAR cache, 2D cache of {n} images, camera files and LAS ready ({(time.time() - t0) / 60:.1f} min)", flush=True)
    # (a) the 2D export of today against the cache the validation numbers were made with
    ids = cached_image_ids('/content/cache2d', z)[:6]
    pats = [f'{z}/images/{i}.JPG' for i in ids]
    code = ("from huggingface_hub import snapshot_download; "
            f"snapshot_download('heig-vd-geo/GridNet-HD', repo_type='dataset', allow_patterns={pats!r}, local_dir='/content/check/data')")
    assert run([PY, '-c', code], show=lambda l: False) == 0, "image download failed"
    assert export2d(z, data='/content/check/data', out='/content/check/cache2d') == 0, "2D export failed"
    agree, mad = [], []
    for i in ids:
        a = np.load(f'/content/check/cache2d/{z}/{i}.npz')['prob']; b = np.load(f'/content/cache2d/{z}/{i}.npz')['prob']
        if b.dtype != np.uint8: b = np.clip(np.round(b.astype(np.float32) * 255), 0, 255).astype(np.uint8)
        agree.append(float((a.argmax(0) == b.argmax(0)).mean())); mad.append(float(np.abs(a.astype(np.int16) - b.astype(np.int16)).mean()) / 255)
    check['export2d'] = {'images': len(ids), 'same_class': float(np.mean(agree)), 'mean_abs_prob_diff': float(np.mean(mad))}
    print(f"[check {z}] 2D export of {len(ids)} images vs the stored cache: same class in {100 * np.mean(agree):.3f} % of the pixels, "
          f"mean |difference| of the probabilities {np.mean(mad):.5f}", flush=True)
    # (b) prediction: must reproduce the full-zone evaluation of the head on this zone
    assert predict(z, '/content/check/pred') == 0, "prediction failed"
    rec = json.load(open(f'/content/check/pred/{z}/predict.json'))
    got = {m: rec[m]['metrics']['mIoU_present'] for m in ('fused', 'lidar')}
    ev = json.load(open(f'{HEAD_EVAL}/eval.json')) if os.path.exists(f'{HEAD_EVAL}/eval.json') else None
    want = None
    if ev and z in ev.get('per_zone', {}):
        want = {'fused': ev['per_zone'][z]['mIoU'], 'lidar': ev['per_zone'][z]['lidar_only_mIoU']}
    old = f'{HEAD_EVAL}/{z}/fused_pred.npy'
    same = float((np.load(old) == np.load(f'/content/check/pred/{z}/fused_pred.npy')).mean()) if os.path.exists(old) else None
    check.update(predicted=got, validation_run=want, same_voxels=same, minutes_predict=rec['minutes'])
    for m in ('fused', 'lidar'):
        print(f"[check {z}] {m:5s} mIoU per voxel {got[m]:.2f}" + (f"   validation run {want[m]:.2f}   difference {got[m] - want[m]:+.2f}" if want else "   (no validation run to compare with)"))
    if same is not None: print(f"[check {z}] fused prediction identical to the validation run on {100 * same:.3f} % of the voxels")
    # (c) leaderboard files + score on the raw points of the LAS (what the leaderboard measures)
    for m in ('fused', 'lidar'):
        assert export(z, '/content/check/pred', m, f'/content/check/{m}', las_root='/content/data') == 0, "export failed"
        r = json.load(open(f'/content/check/{m}/manifest.json'))['zones'][z]
        check[m] = {'points': r['points'], 'voxels': r['voxels'], 'voxel_mIoU': r['voxel_metrics']['mIoU_present'], 'raw_mIoU': r['raw_metrics']['mIoU_present'],
                    'voxel_OA': r['voxel_metrics']['OA'], 'raw_OA': r['raw_metrics']['OA'], 'raw_per_class': r['raw_metrics']['per_class'],
                    'voxel_per_class': r['voxel_metrics']['per_class']}
        print(f"[check {z}] {m:5s} per voxel {check[m]['voxel_mIoU']:.2f} mIoU / {check[m]['voxel_OA']:.2f} OA   per raw point {check[m]['raw_mIoU']:.2f} mIoU / "
              f"{check[m]['raw_OA']:.2f} OA   ({r['points']:,} points, {r['voxels']:,} voxels)")
    check['minutes'] = (time.time() - t0) / 60
    json.dump(check, open(f'{OUT}/check_{z}.json', 'w'), indent=1)
    clean(z); shutil.rmtree('/content/check', ignore_errors=True)
    bad = [m for m in ('fused', 'lidar') if want and abs(got[m] - want[m]) > CHECK_TOLERANCE]
    if bad:
        raise RuntimeError(f"the check zone does not reproduce its validation numbers ({bad}): the test zones are NOT predicted. "
                           "Send this output; to go on anyway raise CHECK_TOLERANCE or empty CHECK_ZONE")
    print(f"[check {z}] passed in {check['minutes']:.1f} min -> {OUT}/check_{z}.json")
else:
    print("CHECK_ZONE is empty: no check")


In [ ]:
#@title 7. Test zones, one at a time (resumable): images + poses from Hugging Face -> 2D cache -> prediction -> leaderboard files on Drive
failed, timing = {}, {}
for z in zones:
    if all(os.path.exists(f'{OUT}/{m}/{z}.npz') for m in ('fused', 'lidar')):
        print(f"[{z}] files exist on Drive (skipped)"); continue
    t0 = time.time(); last = [t0]; step = {}
    def lap(name):
        now = time.time(); step[name] = (now - last[0]) / 60; last[0] = now
    try:
        local_cache(z); lap('cache')
        assert fetch(z, ('images', 'pose')) == 0, "download from Hugging Face failed"
        n_img = len([f for f in os.listdir(f'/content/data/{z}/images') if f.upper().endswith('.JPG')]); lap('download')
        print(f"[{z}] {n_img} images downloaded in {step['download']:.1f} min -> 2D cache", flush=True)
        assert export2d(z) == 0, "2D export failed"; lap('export2d')
        assert predict(z, f'{OUT}/pred') == 0, "prediction failed"; lap('predict')
        for m in ('fused', 'lidar'):
            assert export(z, f'{OUT}/pred', m, f'{OUT}/{m}') == 0, f"export of the {m} file failed"
        lap('files')
    except AssertionError as e:
        failed[z] = str(e); print(f"[{z}] FAILED: {e}  (its local files are kept: run the cell again to retry)", flush=True)
        continue
    clean(z); timing[z] = step
    print(f"[{z}] done in {(time.time() - t0) / 60:.1f} min  (" + ', '.join(f"{k} {v:.1f}" for k, v in step.items()) + ")", flush=True)
if timing: json.dump(timing, open(f'{OUT}/timing_{int(time.time())}.json', 'w'), indent=1)
print("FAILED zones:", failed if failed else "none")


In [ ]:
#@title 8. Summary: what is on Drive and what to upload
# LAS points of every test zone, read when the caches were built (report.txt in GridNet/cache/<zone>/, 2026-11-14).
# Their sum, 766,947,526, is 1.0 % more than the 759,222,245 points the dataset card gives for the test split (reason
# not known). A submission needs one value per LAS point, so the LAS counts are what is checked, zone by zone.
EXPECTED_POINTS = {'t1z4': 49_527_865, 't1z5a': 71_040_345, 't1z7': 122_763_604, 't3z1': 59_879_633, 't3z2': 67_365_114,
                   't3z5': 89_594_873, 't5a2': 95_358_184, 't6z1': 107_067_772, 't6z5': 104_350_136}
ALL_TEST = list(EXPECTED_POINTS)
complete, missing, differs = {}, {}, {}
for m in ('lidar', 'fused'):
    path = f'{OUT}/{m}/manifest.json'
    if not os.path.exists(path):
        print(f"{m}: no file yet"); continue
    man = json.load(open(path)); zs = man['zones']
    print(f"=== {m}  ({man.get('name', '')})  ->  {OUT}/{m}/ ===")
    print(f"{'zone':7s}{'points':>14s}{'voxels':>14s}{'MB':>7s}   predicted classes [%]  " + ' '.join(f"{c[:6]:>6s}" for c in man['class_names']))
    tot, missing[m], differs[m] = 0, [], []
    for z in ALL_TEST:
        if z not in zs or not os.path.exists(f'{OUT}/{m}/{z}.npz'):
            missing[m].append(z); print(f"{z:7s}   MISSING"); continue
        r = zs[z]; tot += r['points']
        if r['points'] != EXPECTED_POINTS[z]: differs[m].append(z)
        print(f"{z:7s}{r['points']:14,d}{r['voxels']:14,d}{r['bytes'] / 1e6:7.1f}" + ' ' * 26 + ' '.join(f"{100 * c / r['points']:6.2f}" for c in r['counts'])
              + ("" if r['points'] == EXPECTED_POINTS[z] else f"   <- the LAS of this zone has {EXPECTED_POINTS[z]:,} points"))
    complete[m] = not missing[m] and not differs[m]
    print(f"{len(ALL_TEST) - len(missing[m])}/{len(ALL_TEST)} zones, {tot:,} points" + (" = one prediction for every LAS point of the nine test zones" if complete[m]
          else f"   (the nine LAS files hold {sum(EXPECTED_POINTS.values()):,} points)"))
    print()
if any(differs.values()):
    print("STOP: the number of predictions differs from the number of LAS points in", {m: z for m, z in differs.items() if z})
    print("Do not submit these files. Find the cause first.")
elif all(complete.get(m) for m in ('lidar', 'fused')):
    print("Both sets are complete. Check the output above first; then upload the 9 files of ONE folder per submission")
    print("(lidar/ first: it checks the whole export path. Published LiDAR-only baselines: SPT 66.91, PTv3 69.32; an unpublished")
    print(" leaderboard entry named Lite-S-test has 72.88. Then fused/.)")
    print("at https://huggingface.co/spaces/heig-vd-geo/GridNet-HD-Leaderboard -> Sign in -> Submit Eval; leave 'Remap classes' UNCHECKED.")
else:
    print("NOT complete yet -> run cell 7 again (finished zones are skipped)")


In [ ]:
#@title 9. (optional) Shut the runtime down once everything above has finished - saves compute units on unattended runs
if SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    print("flushing pending writes to Drive ..."); drive.flush_and_unmount()
    print("results are on Drive; shutting the runtime down"); runtime.unassign()
else:
    print("SHUTDOWN_WHEN_DONE is off -> remember to Disconnect and delete runtime")


## หมายเหตุ
- รูปแบบไฟล์ของ leaderboard: หนึ่งไฟล์ต่อ test zone ชื่อ `<zone>.npz` มี array ชื่อ `data` (uint8) = class ของ **ทุกจุดใน LAS ตามลำดับในไฟล์**
  (เหมือนที่ `create_npz` ในหน้า leaderboard เขียน: `np.savez_compressed(path, data=...)`) · เราส่ง class 0–10 ที่ remap แล้ว จึงไม่ติ๊ก "Remap classes"
- จาก voxel ไปจุดดิบ: โมเดลทำนายหนึ่ง class ต่อ voxel 5 cm; `points.npz` เก็บ `inverse` (จุดดิบ → voxel) ไว้ตั้งแต่ตอนสร้าง cache → จุดดิบทุกจุดได้ class ของ voxel ของมัน
  ไฟล์ถูกอ่านกลับมาตรวจทุกครั้ง (จำนวนจุด, dtype, ช่วงของ class) และ cell 8 เทียบจำนวนจุดของแต่ละโซนกับจำนวนจุดใน LAS ของโซนนั้น (รวม 766,947,526 จุด)
- LiDAR-only กับ fused มาจาก forward pass ชุดเดียวกัน (window, seed, vote เหมือน full-zone evaluation ของ `late_a` ที่ได้ 82.06 บน val)
- `GridNet/leaderboard/<TAG>/pred/<zone>/` เก็บ prediction ระดับ voxel (`fused_pred.npy`, `lidar_pred.npy`) + `predict.json` ไว้ใช้ทำรูป qualitative ภายหลัง
- cell 6 เขียน `check_<zone>.json`: ตัวเลขต่อ voxel เทียบกับต่อจุดดิบของ val zone = ช่องว่างที่ควรคาดไว้ระหว่างตัวเลข validation ของเรากับตัวเลขบน leaderboard (นอกเหนือจากการที่เป็นคนละโซน)
- ถ้า session หลุด: Run all ใหม่ได้เลย โซนที่มีไฟล์บน Drive แล้วถูกข้าม (cell 6 จะรันซ้ำ ~10 นาที; ตั้ง `CHECK_ZONE` ว่างเพื่อข้าม)
- ส่ง leaderboard แค่ 2 ครั้งตามแผน: LiDAR-only (sanity) และ fused model ตัวสุดท้าย
- จบแล้ว **Disconnect and delete runtime**